# Gold Layer — dimensions (Kimball, design 1.3)

Build `dim_date`, `dim_video`, `dim_channel`, `dim_commenter`, `dim_keyword` trong `prod.gold` từ Silver `prod.silver.stg_*`.

**Load patterns (design 1.5):**
- `dim_date`: **overwrite** (generated calendar 2005 → 2030 (YouTube launch year, so comments and videos older than 2020 still join) + VN holiday seed (lunar holidays 2020+ only)), atomic `CREATE OR REPLACE`.
- `dim_video`, `dim_channel`, `dim_commenter`, `dim_keyword`: **SCD 1 MERGE** on the natural key, guarded by `record_checksum` so unchanged rows are not rewritten and `etl_date` still means "changed". `last_seen_at` is refreshed without touching `etl_date`.

**Keys:** surrogate key = `xxhash64(natural key)` (deterministic, facts compute it without a lookup). Every dimension has an **Unknown** row with key **−1** (natural key NULL); facts with no match point to it.

Counters (`view_count`, `subscriber_count`, ...) are **not** here: they live in the snapshot facts.


In [ ]:
%run ./_udfs


In [ ]:
# --- Config ---
# --- Layer schemas: job parameters bronze / silver / gold (catalog.schema), default prod.* ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

SNAPSHOT_TZ = "Asia/Ho_Chi_Minh"  # every date key uses Vietnam time

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")
print(f"Silver source: {SILVER}.stg_*")
print(f"Gold target: {GOLD}.dim_*")


## dim_date

SCD 0, generated. `is_vn_holiday` / `holiday_name` come from the seed list below: fixed-date public holidays plus Tết (mùng 1) and Giỗ Tổ Hùng Vương (10/3 âm lịch) converted to solar dates. Review the seed when the holiday calendar changes, then rerun (overwrite).


In [ ]:
# --- VN holiday seed (lunar holidays as solar dates) ---
FIXED_HOLIDAYS = {"01-01": "Tết Dương lịch", "04-30": "Ngày Giải phóng miền Nam", "05-01": "Quốc tế Lao động", "09-02": "Quốc khánh"}
TET = ["2020-01-25", "2021-02-12", "2022-02-01", "2023-01-22", "2024-02-10", "2025-01-29",
       "2026-02-17", "2027-02-06", "2028-01-26", "2029-02-13", "2030-02-03"]
HUNG_KINGS = ["2020-04-02", "2021-04-21", "2022-04-10", "2023-04-29", "2024-04-18", "2025-04-07",
              "2026-04-26", "2027-04-16", "2028-04-04", "2029-04-23", "2030-04-12"]

seed_rows = [(f"{y}-{md}", name) for y in range(2005, 2031) for md, name in FIXED_HOLIDAYS.items()]
seed_rows += [(d, "Tết Nguyên Đán") for d in TET] + [(d, "Giỗ Tổ Hùng Vương") for d in HUNG_KINGS]
spark.createDataFrame(seed_rows, "holiday_date STRING, holiday_name STRING").createOrReplaceTempView("vn_holidays")

spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD}.dim_date
    COMMENT 'Calendar dimension 2005-2030 (SCD 0). date_key = yyyymmdd, -1 = Unknown.'
    AS
    WITH days AS (
        SELECT explode(sequence(DATE'2005-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS full_date
    )
    SELECT
        CAST(date_format(d.full_date, 'yyyyMMdd') AS INT)      AS date_key,
        d.full_date,
        CAST(day(d.full_date) AS TINYINT)                       AS day_of_month,
        CAST(weekday(d.full_date) + 1 AS TINYINT)               AS day_of_week,
        date_format(d.full_date, 'EEEE')                        AS day_name,
        weekday(d.full_date) + 1 IN (6, 7)                      AS is_weekend,
        CAST(weekofyear(d.full_date) AS TINYINT)                AS week_of_year,
        CAST(month(d.full_date) AS TINYINT)                     AS month,
        date_format(d.full_date, 'MMMM')                        AS month_name,
        CAST(quarter(d.full_date) AS TINYINT)                   AS quarter,
        CAST(year(d.full_date) AS SMALLINT)                     AS year,
        date_format(d.full_date, 'yyyy-MM')                     AS year_month,
        h.holiday_name IS NOT NULL                              AS is_vn_holiday,
        h.holiday_name
    FROM days d
    LEFT JOIN (
        SELECT CAST(holiday_date AS DATE) AS holiday_date, concat_ws(' / ', collect_list(holiday_name)) AS holiday_name
        FROM vn_holidays GROUP BY 1
    ) h ON h.holiday_date = d.full_date
    UNION ALL
    SELECT -1, NULL, NULL, NULL, 'Unknown', NULL, NULL, NULL, 'Unknown', NULL, NULL, 'Unknown', false, NULL
""")
print(f"✅ {GOLD}.dim_date: {spark.table(f'{GOLD}.dim_date').count()} rows")


## dim_video

SCD 1 · one row per `video_id` · latest snapshot per video from `stg_videos`. `category_name` from the YouTube category list (videoCategories.list, region VN), `duration_seconds` parsed from ISO 8601. `is_available` = the video was returned on the latest refresh day.

`video_type` / `video_type_confidence` / `is_food_review` come from `stg_video_classification` (LLM on title + description). `is_food_review = false` only for confident `not_food`; serving tasks (Gemini transcript, restaurant extraction) skip those videos. Not yet classified = `true`.


In [ ]:
# --- YouTube video categories (videoCategories.list) ---
CATEGORIES = [("1", "Film & Animation"), ("2", "Autos & Vehicles"), ("10", "Music"), ("15", "Pets & Animals"),
              ("17", "Sports"), ("19", "Travel & Events"), ("20", "Gaming"), ("22", "People & Blogs"),
              ("23", "Comedy"), ("24", "Entertainment"), ("25", "News & Politics"), ("26", "Howto & Style"),
              ("27", "Education"), ("28", "Science & Technology"), ("29", "Nonprofits & Activism")]
spark.createDataFrame(CATEGORIES, "category_id STRING, category_name STRING").createOrReplaceTempView("yt_categories")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {GOLD}.dim_video (
        video_key          BIGINT,
        video_id           STRING,
        title              STRING,
        description        STRING,
        tags               ARRAY<STRING>,
        category_id        STRING,
        category_name      STRING,
        duration_iso       STRING,
        duration_seconds   INT,
        definition         STRING,
        has_caption        BOOLEAN,
        thumbnail_url      STRING,
        video_url          STRING,
        channel_id         STRING,
        published_at       TIMESTAMP,
        published_date_key INT,
        is_available       BOOLEAN,
        video_type         STRING,
        video_type_confidence DOUBLE,
        is_food_review     BOOLEAN,
        first_seen_at      TIMESTAMP,
        last_seen_at       TIMESTAMP,
        record_checksum    STRING,
        etl_date           TIMESTAMP
    )
    USING DELTA
    COMMENT 'Video dimension (SCD 1). video_key = xxhash64(video_id), -1 = Unknown.'
""")

# --- Video type (stg_video_classification, LLM): added after the first release, so add the columns if missing ---
existing_cols = {f.name for f in spark.table(f"{GOLD}.dim_video").schema}
if "video_type" not in existing_cols:
    spark.sql(f"""ALTER TABLE {GOLD}.dim_video ADD COLUMNS (
        video_type STRING AFTER is_available, video_type_confidence DOUBLE AFTER video_type,
        is_food_review BOOLEAN AFTER video_type_confidence)""")
CLASS_TABLE = f"{SILVER}.stg_video_classification"
if spark.catalog.tableExists(CLASS_TABLE):
    CLASS_SQL = f"SELECT video_id, video_type, confidence, is_food_review FROM {CLASS_TABLE}"
else:  # before the first stg_video_classification run
    CLASS_SQL = ("SELECT CAST(NULL AS STRING) AS video_id, CAST(NULL AS STRING) AS video_type, "
                 "CAST(NULL AS DOUBLE) AS confidence, CAST(NULL AS BOOLEAN) AS is_food_review WHERE false")

spark.sql(rf"""
    CREATE OR REPLACE TEMP VIEW dim_video_src AS
    WITH ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY video_id ORDER BY snapshot_date DESC, updated_at DESC) AS rn
        FROM {SILVER}.stg_videos
    ),
    seen AS (
        SELECT video_id, MIN(created_at) AS first_seen_at, MAX(COALESCE(updated_at, created_at)) AS last_seen_at,
               MAX(snapshot_date) AS last_snapshot
        FROM {SILVER}.stg_videos GROUP BY video_id
    ),
    latest_refresh AS (SELECT MAX(snapshot_date) AS d FROM {SILVER}.stg_videos),
    base AS (
        SELECT
            xxhash64(r.video_id)                                         AS video_key,
            r.video_id, r.title, r.description, r.tags, r.category_id,
            c.category_name,
            r.duration                                                   AS duration_iso,
            CASE WHEN r.duration IS NOT NULL THEN
                3600 * COALESCE(CAST(NULLIF(regexp_extract(r.duration, '(\\d+)H', 1), '') AS INT), 0)
              +   60 * COALESCE(CAST(NULLIF(regexp_extract(r.duration, '(\\d+)M', 1), '') AS INT), 0)
              +        COALESCE(CAST(NULLIF(regexp_extract(r.duration, '(\\d+)S', 1), '') AS INT), 0)
            END                                                          AS duration_seconds,
            r.definition, r.has_caption, r.thumbnail_url, r.video_url, r.channel_id, r.published_at,
            CAST(date_format(from_utc_timestamp(r.published_at, '{SNAPSHOT_TZ}'), 'yyyyMMdd') AS INT) AS published_date_key,
            s.last_snapshot = lr.d                                       AS is_available,
            vc.video_type,
            vc.confidence                                                AS video_type_confidence,
            coalesce(vc.is_food_review, true)                            AS is_food_review,
            s.first_seen_at, s.last_seen_at
        FROM ranked r
        JOIN seen s ON s.video_id = r.video_id
        CROSS JOIN latest_refresh lr
        LEFT JOIN yt_categories c ON c.category_id = r.category_id
        LEFT JOIN ({CLASS_SQL}) vc ON vc.video_id = r.video_id
        WHERE r.rn = 1
    )
    SELECT *,
        md5(concat_ws('||', title, description, array_join(tags, ','), category_id, category_name, duration_iso,
                      definition, CAST(has_caption AS STRING), thumbnail_url, video_url, channel_id,
                      CAST(published_at AS STRING), CAST(is_available AS STRING),
                      video_type, CAST(video_type_confidence AS STRING), CAST(is_food_review AS STRING))) AS record_checksum,
        current_timestamp() AS etl_date
    FROM base
""")

spark.sql(f"""
    MERGE INTO {GOLD}.dim_video AS t
    USING dim_video_src AS s ON t.video_id = s.video_id
    WHEN MATCHED AND t.record_checksum <> s.record_checksum THEN UPDATE SET *
    WHEN MATCHED AND NOT (t.last_seen_at <=> s.last_seen_at) THEN UPDATE SET t.last_seen_at = s.last_seen_at
    WHEN NOT MATCHED THEN INSERT *
""")
spark.sql(f"""
    INSERT INTO {GOLD}.dim_video (video_key, title, category_name, is_available, etl_date)
    SELECT -1, 'Unknown', 'Unknown', false, current_timestamp()
    WHERE NOT EXISTS (SELECT 1 FROM {GOLD}.dim_video WHERE video_key = -1)
""")
print(f"✅ {GOLD}.dim_video: {spark.table(f'{GOLD}.dim_video').count()} rows")


## dim_channel

SCD 1 · one row per `channel_id` · latest snapshot per channel from `stg_channels`. `created_date_key` = channel creation date (Vietnam time).


In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {GOLD}.dim_channel (
        channel_key         BIGINT,
        channel_id          STRING,
        channel_title       STRING,
        channel_description STRING,
        custom_url          STRING,
        country             STRING,
        default_language    STRING,
        thumbnail_url       STRING,
        published_at        TIMESTAMP,
        created_date_key    INT,
        is_available        BOOLEAN,
        first_seen_at       TIMESTAMP,
        last_seen_at        TIMESTAMP,
        record_checksum     STRING,
        etl_date            TIMESTAMP
    )
    USING DELTA
    COMMENT 'Publishing channel dimension (SCD 1). channel_key = xxhash64(channel_id), -1 = Unknown.'
""")

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW dim_channel_src AS
    WITH ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY channel_id ORDER BY snapshot_date DESC, updated_at DESC) AS rn
        FROM {SILVER}.stg_channels
    ),
    seen AS (
        SELECT channel_id, MIN(created_at) AS first_seen_at, MAX(COALESCE(updated_at, created_at)) AS last_seen_at,
               MAX(snapshot_date) AS last_snapshot
        FROM {SILVER}.stg_channels GROUP BY channel_id
    ),
    latest_refresh AS (SELECT MAX(snapshot_date) AS d FROM {SILVER}.stg_channels),
    base AS (
        SELECT
            xxhash64(r.channel_id) AS channel_key,
            r.channel_id, r.channel_title, r.channel_description, r.custom_url, r.country, r.default_language,
            r.thumbnail_url, r.published_at,
            CAST(date_format(from_utc_timestamp(r.published_at, '{SNAPSHOT_TZ}'), 'yyyyMMdd') AS INT) AS created_date_key,
            s.last_snapshot = lr.d AS is_available,
            s.first_seen_at, s.last_seen_at
        FROM ranked r
        JOIN seen s ON s.channel_id = r.channel_id
        CROSS JOIN latest_refresh lr
        WHERE r.rn = 1
    )
    SELECT *,
        md5(concat_ws('||', channel_title, channel_description, custom_url, country, default_language, thumbnail_url,
                      CAST(published_at AS STRING), CAST(is_available AS STRING))) AS record_checksum,
        current_timestamp() AS etl_date
    FROM base
""")

spark.sql(f"""
    MERGE INTO {GOLD}.dim_channel AS t
    USING dim_channel_src AS s ON t.channel_id = s.channel_id
    WHEN MATCHED AND t.record_checksum <> s.record_checksum THEN UPDATE SET *
    WHEN MATCHED AND NOT (t.last_seen_at <=> s.last_seen_at) THEN UPDATE SET t.last_seen_at = s.last_seen_at
    WHEN NOT MATCHED THEN INSERT *
""")
spark.sql(f"""
    INSERT INTO {GOLD}.dim_channel (channel_key, channel_title, is_available, etl_date)
    SELECT -1, 'Unknown', false, current_timestamp()
    WHERE NOT EXISTS (SELECT 1 FROM {GOLD}.dim_channel WHERE channel_key = -1)
""")
print(f"✅ {GOLD}.dim_channel: {spark.table(f'{GOLD}.dim_channel').count()} rows")


## dim_commenter

SCD 1 · one row per comment author (a viewer, separate from the publishing channel). Key works even when `author_channel_id` is missing: `xxhash64(COALESCE(author_channel_id, 'name:' || author_display_name))`.


In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {GOLD}.dim_commenter (
        commenter_key       BIGINT,
        author_channel_id   STRING,
        author_display_name STRING,
        first_comment_at    TIMESTAMP,
        last_comment_at     TIMESTAMP,
        record_checksum     STRING,
        etl_date            TIMESTAMP
    )
    USING DELTA
    COMMENT 'Comment author dimension (SCD 1). commenter_key = xxhash64(author_channel_id or name), -1 = Unknown.'
""")

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW dim_commenter_src AS
    WITH c AS (
        SELECT *, COALESCE(author_channel_id, 'name:' || author_display_name) AS commenter_nk
        FROM {SILVER}.stg_comments
        WHERE COALESCE(author_channel_id, author_display_name) IS NOT NULL
    ),
    ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY commenter_nk ORDER BY published_at DESC) AS rn FROM c
    ),
    agg AS (
        SELECT commenter_nk, MIN(published_at) AS first_comment_at, MAX(published_at) AS last_comment_at
        FROM c GROUP BY commenter_nk
    )
    SELECT
        xxhash64(r.commenter_nk) AS commenter_key,
        r.author_channel_id, r.author_display_name, a.first_comment_at, a.last_comment_at,
        md5(concat_ws('||', r.author_channel_id, r.author_display_name,
                      CAST(a.first_comment_at AS STRING), CAST(a.last_comment_at AS STRING))) AS record_checksum,
        current_timestamp() AS etl_date
    FROM ranked r JOIN agg a ON a.commenter_nk = r.commenter_nk
    WHERE r.rn = 1
""")

spark.sql(f"""
    MERGE INTO {GOLD}.dim_commenter AS t
    USING dim_commenter_src AS s ON t.commenter_key = s.commenter_key
    WHEN MATCHED AND t.record_checksum <> s.record_checksum THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
spark.sql(f"""
    INSERT INTO {GOLD}.dim_commenter (commenter_key, author_display_name, etl_date)
    SELECT -1, 'Unknown', current_timestamp()
    WHERE NOT EXISTS (SELECT 1 FROM {GOLD}.dim_commenter WHERE commenter_key = -1)
""")
print(f"✅ {GOLD}.dim_commenter: {spark.table(f'{GOLD}.dim_commenter').count()} rows")


## dim_keyword

One row per **normalized** search keyword: "Phở ngon Sài Gòn", "phở ngon sài gòn " → `phở ngon sài gòn` (diacritics kept). `dish` / `city` stay NULL until keyword parsing is built.


In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {GOLD}.dim_keyword (
        keyword_key        BIGINT,
        keyword_normalized STRING,
        keyword_unaccented STRING,
        keyword_raw_first  STRING,
        raw_variant_count  INT,
        dish               STRING,
        city               STRING,
        first_searched_at  TIMESTAMP,
        record_checksum    STRING,
        etl_date           TIMESTAMP
    )
    USING DELTA
    COMMENT 'Normalized search keyword dimension. keyword_key = xxhash64(keyword_normalized), -1 = Unknown.'
""")

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW dim_keyword_src AS
    WITH k AS (
        SELECT normalize_keyword(search_keyword) AS keyword_normalized, search_keyword, fetched_at
        FROM {SILVER}.stg_search_results
        WHERE search_keyword IS NOT NULL AND trim(search_keyword) <> ''
    ),
    first_raw AS (
        SELECT keyword_normalized, search_keyword AS keyword_raw_first,
               ROW_NUMBER() OVER (PARTITION BY keyword_normalized ORDER BY fetched_at) AS rn
        FROM k
    ),
    agg AS (
        SELECT keyword_normalized, CAST(COUNT(DISTINCT search_keyword) AS INT) AS raw_variant_count,
               MIN(fetched_at) AS first_searched_at
        FROM k GROUP BY keyword_normalized
    )
    SELECT
        xxhash64(a.keyword_normalized) AS keyword_key,
        a.keyword_normalized,
        unaccent_vi(a.keyword_normalized) AS keyword_unaccented,
        f.keyword_raw_first,
        a.raw_variant_count,
        CAST(NULL AS STRING) AS dish,
        CAST(NULL AS STRING) AS city,
        a.first_searched_at,
        md5(concat_ws('||', f.keyword_raw_first, CAST(a.raw_variant_count AS STRING), CAST(a.first_searched_at AS STRING))) AS record_checksum,
        current_timestamp() AS etl_date
    FROM agg a JOIN first_raw f ON f.keyword_normalized = a.keyword_normalized AND f.rn = 1
""")

spark.sql(f"""
    MERGE INTO {GOLD}.dim_keyword AS t
    USING dim_keyword_src AS s ON t.keyword_normalized = s.keyword_normalized
    WHEN MATCHED AND t.record_checksum <> s.record_checksum THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
spark.sql(f"""
    INSERT INTO {GOLD}.dim_keyword (keyword_key, keyword_normalized, etl_date)
    SELECT -1, NULL, current_timestamp()
    WHERE NOT EXISTS (SELECT 1 FROM {GOLD}.dim_keyword WHERE keyword_key = -1)
""")
print(f"✅ {GOLD}.dim_keyword: {spark.table(f'{GOLD}.dim_keyword').count()} rows")


In [ ]:
# --- Verify ---
for t in ["dim_date", "dim_video", "dim_channel", "dim_commenter", "dim_keyword"]:
    n = spark.table(f"{GOLD}.{t}").count()
    print(f"📊 {GOLD}.{t}: {n} rows")
print("\n--- Gold dims complete ---")
